# Lecture : Graph-based Visualization

## Lab 06 : Uniform Manifold Approximation and Projection (UMAP)

### Xavier Bresson  


In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/04_Visualization'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pip install umap-learn
    !pwd
    

In [ ]:
# Load libraries
import numpy as np
import scipy.io
from matplotlib import pyplot
import matplotlib.pyplot as plt
import time
import sys; sys.path.insert(0, 'lib/')
import scipy.sparse.linalg
# import scipy.ndimage
from lib.utils import construct_knn_graph, nldr_visualization
import warnings; warnings.filterwarnings("ignore")
from sklearn.manifold import TSNE
import umap
import torch, torchvision
import torchvision.transforms as transforms
import numpy as np
import os


# UMAP for MNIST

In [ ]:
# MNIST dataset
mat = scipy.io.loadmat('datasets/MNIST_data.mat')
X = Xnumpy = mat['X']
n = X.shape[0]
d = X.shape[1]
C = mat['C'].squeeze()
print(n,d)


## Question: explore UMAP
#### 1. UMAP is a graph-based algorithm. True or False?

#### 2. How the n_neighbors parameter affect the visualization?


#### 3. Compared to UMAP and TSNE in terms of local / global structure 



In [ ]:
# UMAP : https://umap-learn.readthedocs.io
start = time.time()
# reducer = umap.UMAP(n_components=3, n_neighbors=20, random_state=42, transform_seed=42, verbose=False) # UMAP
n=5 # try different value between 2~200
reducer = umap.UMAP(n_components=3, n_neighbors=n, verbose=True)
print(X.shape)
embedding = reducer.fit_transform(X)
print('time(sec):',(time.time()-start)/1)

print(embedding.shape)
Xvis = embedding[:,0]
Yvis = embedding[:,1]
Zvis = embedding[:,2]

# 2D Visualization
plt.figure(3)
plt.scatter(Xvis, Yvis, c=C, s=1, color=pyplot.jet())
plt.title('MNIST visualized with UMAP') 
plt.show()


In [ ]:
# 3D Visualization
import plotly.graph_objects as go
data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1)) # data as points
# data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1, showscale=True)) # w/ colobar 
fig = go.Figure(data=[data]) 
fig.update_layout(margin=dict(l=0, r=0, b=0, t=25, pad=0)) # tight layout but t=25 required for showing title 
fig.update_layout(autosize=False, width=800, height=800, title_text="MNIST visualized with UMAP") # figure size and title
# fig.update_layout(scene = dict(xaxis = dict(visible=False), yaxis = dict(visible=False), zaxis = dict(visible=False))) # no grid, no axis 
# fig.update_layout(scene = dict(xaxis_title = ' ', yaxis_title = ' ', zaxis_title = ' ')) # no axis name 
fig.update_layout(scene = dict(zaxis = dict(showgrid = True, showticklabels = False), zaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(yaxis = dict(showgrid = True, showticklabels = False), yaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(xaxis = dict(showgrid = True, showticklabels = False), xaxis_title = ' ') ) # no range values, no axis name, grid on
fig.show()


# UMAP for CIFAR

In [ ]:
# CIFAR 
if not os.path.isfile('datasets/cifar.pt'):  # download and prepare CIFAR dataset
    trainset = torchvision.datasets.CIFAR10(root='datasets/', train=True, download=True, transform=transforms.ToTensor())
    print('num_data : ',len(trainset))
    print('data, label : ',trainset[0][0].size(),trainset[0][1])
    train_data = torch.Tensor(50000,3,32,32)
    train_label = torch.LongTensor(50000)
    for idx, data in enumerate(trainset):
        train_data[idx] = data[0]
        train_label[idx] = data[1]
    torch.save([train_data, train_label],'datasets/cifar.pt')
else:
    train_data, train_label = torch.load('datasets/cifar.pt')
    
X, C = train_data, train_label
print('X, C :',X.size(), C.size())
X = X.view(50000,-1).numpy()
C = C.numpy()


In [ ]:
# print one CIFAR image
idx = 2
pic, label = train_data[idx], train_label[idx]
print(pic.size())
print('min=',torch.min(pic), '  max=',torch.max(pic) )
plt.imshow( np.transpose(  pic.numpy() , (1, 2, 0))  )
plt.show()
print(label)


In [ ]:
# UMAP : https://umap-learn.readthedocs.io
start = time.time()
# reducer = umap.UMAP(n_components=3, n_neighbors=20, random_state=42, transform_seed=42, verbose=False) # UMAP
reducer = umap.UMAP(n_components=3, verbose=True)
print(X.shape)
embedding = reducer.fit_transform(X)
print('time(sec):',(time.time()-start)/1)

print(embedding.shape)
Xvis = embedding[:,0]
Yvis = embedding[:,1]
Zvis = embedding[:,2]

# 2D Visualization
plt.figure(3)
plt.scatter(Xvis, Yvis, c=C, s=1, color=pyplot.jet())
plt.title('CIFAR (raw images) visualized with UMAP') 
plt.show()


In [ ]:
# 3D Visualization
import plotly.graph_objects as go
data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1)) # data as points
# data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1, showscale=True)) # w/ colobar 
fig = go.Figure(data=[data]) 
fig.update_layout(margin=dict(l=0, r=0, b=0, t=25, pad=0)) # tight layout but t=25 required for showing title 
fig.update_layout(autosize=False, width=800, height=800, title_text="CIFAR (raw images) visualized with UMAP") # figure size and title
# fig.update_layout(scene = dict(xaxis = dict(visible=False), yaxis = dict(visible=False), zaxis = dict(visible=False))) # no grid, no axis 
# fig.update_layout(scene = dict(xaxis_title = ' ', yaxis_title = ' ', zaxis_title = ' ')) # no axis name 
fig.update_layout(scene = dict(zaxis = dict(showgrid = True, showticklabels = False), zaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(yaxis = dict(showgrid = True, showticklabels = False), yaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(xaxis = dict(showgrid = True, showticklabels = False), xaxis_title = ' ') ) # no range values, no axis name, grid on
fig.show()
